# Exploratory Data Analysis: Facebook Political Ads Dataset

This notebook provides a comprehensive exploratory data analysis of the Facebook political ads dataset, with a focus on identifying patterns and features relevant for detecting manipulation.

## Dataset Overview
- **Source**: Facebook Ad Library 2022
- **Purpose**: Analyze political advertising patterns for manipulation detection
- **Target**: Build a model to classify manipulative content


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
from datetime import datetime
from collections import Counter
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")


In [ ]:
# Load the dataset
data_path = '../data/fb_2022_adid_var_040825.csv'
df = pd.read_csv(data_path, low_memory=False)

print(f"Dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nColumns: {len(df.columns)}")
print(f"Rows: {len(df):,}")


In [ ]:
# Display basic information about the dataset
print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"\nFirst few rows:")
df.head()


In [ ]:
# Column names and data types
print("=" * 80)
print("COLUMN INFORMATION")
print("=" * 80)
print(f"\nTotal columns: {len(df.columns)}")
print(f"\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:3d}. {col}")

print(f"\n\nData types:")
print(df.dtypes.value_counts())


In [ ]:
# Missing values analysis
print("=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)

missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100,
    'Non_Missing_Count': df.notna().sum()
})

missing_data = missing_data.sort_values('Missing_Percentage', ascending=False)
print(f"\nColumns with missing values: {(missing_data['Missing_Percentage'] > 0).sum()}")
print(f"\nTop 20 columns with most missing values:")
print(missing_data[missing_data['Missing_Percentage'] > 0].head(20).to_string(index=False))

# Visualize missing values
plt.figure(figsize=(14, 8))
top_missing = missing_data[missing_data['Missing_Percentage'] > 0].head(30)
sns.barplot(data=top_missing, y='Column', x='Missing_Percentage', palette='viridis')
plt.title('Top 30 Columns with Missing Values', fontsize=16, fontweight='bold')
plt.xlabel('Missing Percentage (%)', fontsize=12)
plt.ylabel('Column Name', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Helper function to parse JSON bounds
def parse_bounds(value):
    """Extract numeric values from JSON bounds format"""
    if pd.isna(value):
        return None, None
    try:
        if isinstance(value, str):
            data = json.loads(value)
            lower = float(data.get('lower_bound', 0))
            upper = float(data.get('upper_bound', lower))
            return lower, upper
    except:
        pass
    return None, None

# Parse spend and impressions
print("=" * 80)
print("SPENDING AND IMPRESSIONS ANALYSIS")
print("=" * 80)

spend_bounds = df['spend'].apply(parse_bounds)
df['spend_lower'] = spend_bounds.apply(lambda x: x[0] if x[0] is not None else np.nan)
df['spend_upper'] = spend_bounds.apply(lambda x: x[1] if x[1] is not None else np.nan)
df['spend_mid'] = (df['spend_lower'] + df['spend_upper']) / 2

impressions_bounds = df['impressions'].apply(parse_bounds)
df['impressions_lower'] = impressions_bounds.apply(lambda x: x[0] if x[0] is not None else np.nan)
df['impressions_upper'] = impressions_bounds.apply(lambda x: x[1] if x[1] is not None else np.nan)
df['impressions_mid'] = (df['impressions_lower'] + df['impressions_upper']) / 2

# Calculate cost per impression (CPM)
df['cpm'] = (df['spend_mid'] / df['impressions_mid']) * 1000
df['cpm'] = df['cpm'].replace([np.inf, -np.inf], np.nan)

print(f"\nSpending Statistics:")
print(f"  Total ads with spending data: {df['spend_mid'].notna().sum():,}")
print(f"  Mean spending: ${df['spend_mid'].mean():,.2f}")
print(f"  Median spending: ${df['spend_mid'].median():,.2f}")
print(f"  Max spending: ${df['spend_mid'].max():,.2f}")
print(f"  Min spending: ${df['spend_mid'].min():,.2f}")

print(f"\nImpressions Statistics:")
print(f"  Total ads with impressions data: {df['impressions_mid'].notna().sum():,}")
print(f"  Mean impressions: {df['impressions_mid'].mean():,.0f}")
print(f"  Median impressions: {df['impressions_mid'].median():,.0f}")
print(f"  Max impressions: {df['impressions_mid'].max():,.0f}")

print(f"\nCPM Statistics:")
print(f"  Mean CPM: ${df['cpm'].mean():.2f}")
print(f"  Median CPM: ${df['cpm'].median():.2f}")


In [ ]:
# Visualize spending and impressions distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Spending distribution (log scale)
axes[0, 0].hist(df['spend_mid'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Spending ($)', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Spending Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_yscale('log')

# Spending distribution (log scale for x-axis)
axes[0, 1].hist(df['spend_mid'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Spending ($)', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('Spending Distribution (Log Scale)', fontsize=12, fontweight='bold')
axes[0, 1].set_xscale('log')

# Impressions distribution
axes[0, 2].hist(df['impressions_mid'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0, 2].set_xlabel('Impressions', fontsize=11)
axes[0, 2].set_ylabel('Frequency', fontsize=11)
axes[0, 2].set_title('Impressions Distribution', fontsize=12, fontweight='bold')
axes[0, 2].set_yscale('log')

# Impressions distribution (log scale for x-axis)
axes[1, 0].hist(df['impressions_mid'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1, 0].set_xlabel('Impressions', fontsize=11)
axes[1, 0].set_ylabel('Frequency', fontsize=11)
axes[1, 0].set_title('Impressions Distribution (Log Scale)', fontsize=12, fontweight='bold')
axes[1, 0].set_xscale('log')

# CPM distribution
axes[1, 1].hist(df['cpm'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1, 1].set_xlabel('CPM ($)', fontsize=11)
axes[1, 1].set_ylabel('Frequency', fontsize=11)
axes[1, 1].set_title('Cost Per Mille (CPM) Distribution', fontsize=12, fontweight='bold')

# Spending vs Impressions scatter
sample_df = df[df['spend_mid'].notna() & df['impressions_mid'].notna()].sample(min(10000, len(df)))
axes[1, 2].scatter(sample_df['impressions_mid'], sample_df['spend_mid'], alpha=0.3, s=1)
axes[1, 2].set_xlabel('Impressions', fontsize=11)
axes[1, 2].set_ylabel('Spending ($)', fontsize=11)
axes[1, 2].set_title('Spending vs Impressions', fontsize=12, fontweight='bold')
axes[1, 2].set_xscale('log')
axes[1, 2].set_yscale('log')

plt.tight_layout()
plt.show()


In [ ]:
# Temporal analysis
print("=" * 80)
print("TEMPORAL ANALYSIS")
print("=" * 80)

# Parse date columns
df['ad_creation_time'] = pd.to_datetime(df['ad_creation_time'], errors='coerce')
df['ad_delivery_start_time'] = pd.to_datetime(df['ad_delivery_start_time'], errors='coerce')
df['ad_delivery_stop_time'] = pd.to_datetime(df['ad_delivery_stop_time'], errors='coerce')

# Calculate ad duration
df['ad_duration_days'] = (df['ad_delivery_stop_time'] - df['ad_delivery_start_time']).dt.days

# Extract temporal features
df['creation_year'] = df['ad_creation_time'].dt.year
df['creation_month'] = df['ad_creation_time'].dt.month
df['creation_week'] = df['ad_creation_time'].dt.isocalendar().week
df['creation_day_of_week'] = df['ad_creation_time'].dt.dayofweek
df['creation_date'] = df['ad_creation_time'].dt.date

print(f"\nDate Range:")
print(f"  Creation time: {df['ad_creation_time'].min()} to {df['ad_creation_time'].max()}")
print(f"  Delivery start: {df['ad_delivery_start_time'].min()} to {df['ad_delivery_start_time'].max()}")
print(f"  Delivery stop: {df['ad_delivery_stop_time'].min()} to {df['ad_delivery_stop_time'].max()}")

print(f"\nAd Duration Statistics:")
print(f"  Mean duration: {df['ad_duration_days'].mean():.1f} days")
print(f"  Median duration: {df['ad_duration_days'].median():.1f} days")
print(f"  Max duration: {df['ad_duration_days'].max():.0f} days")
print(f"  Min duration: {df['ad_duration_days'].min():.0f} days")


In [ ]:
# Visualize temporal patterns
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Ads created over time
creation_counts = df.groupby('creation_date').size()
axes[0, 0].plot(creation_counts.index, creation_counts.values, linewidth=2)
axes[0, 0].set_xlabel('Date', fontsize=11)
axes[0, 0].set_ylabel('Number of Ads', fontsize=11)
axes[0, 0].set_title('Ads Created Over Time', fontsize=12, fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(True, alpha=0.3)

# Ads by month
month_counts = df['creation_month'].value_counts().sort_index()
axes[0, 1].bar(month_counts.index, month_counts.values, color='steelblue', alpha=0.7)
axes[0, 1].set_xlabel('Month', fontsize=11)
axes[0, 1].set_ylabel('Number of Ads', fontsize=11)
axes[0, 1].set_title('Ads Created by Month', fontsize=12, fontweight='bold')
axes[0, 1].set_xticks(range(1, 13))
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Ads by day of week
dow_counts = df['creation_day_of_week'].value_counts().sort_index()
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[1, 0].bar(range(7), [dow_counts.get(i, 0) for i in range(7)], color='coral', alpha=0.7)
axes[1, 0].set_xticks(range(7))
axes[1, 0].set_xticklabels(dow_names)
axes[1, 0].set_xlabel('Day of Week', fontsize=11)
axes[1, 0].set_ylabel('Number of Ads', fontsize=11)
axes[1, 0].set_title('Ads Created by Day of Week', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Ad duration distribution
axes[1, 1].hist(df['ad_duration_days'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1, 1].set_xlabel('Duration (Days)', fontsize=11)
axes[1, 1].set_ylabel('Frequency', fontsize=11)
axes[1, 1].set_title('Ad Duration Distribution', fontsize=12, fontweight='bold')
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.show()


In [ ]:
# Party classification analysis
print("=" * 80)
print("PARTY CLASSIFICATION ANALYSIS")
print("=" * 80)

party_cols = ['party_all', 'party_all_clf_pdid', 'party_all_clf_adid', 'party_all_clf_adid_agg']
party_data = {}

for col in party_cols:
    if col in df.columns:
        party_data[col] = df[col].value_counts()
        print(f"\n{col}:")
        print(party_data[col])
        print(f"  Missing: {df[col].isna().sum():,} ({df[col].isna().sum()/len(df)*100:.1f}%)")

# Probability columns
prob_cols = ['prob_dem', 'prob_rep', 'prob_other']
for col in prob_cols:
    if col in df.columns:
        print(f"\n{col} Statistics:")
        print(f"  Mean: {df[col].mean():.4f}")
        print(f"  Median: {df[col].median():.4f}")
        print(f"  Std: {df[col].std():.4f}")
        print(f"  Missing: {df[col].isna().sum():,}")


In [ ]:
# Visualize party classifications
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Party distribution (party_all)
if 'party_all' in df.columns:
    party_counts = df['party_all'].value_counts()
    axes[0, 0].bar(party_counts.index, party_counts.values, color=['blue', 'red', 'gray'], alpha=0.7)
    axes[0, 0].set_xlabel('Party', fontsize=11)
    axes[0, 0].set_ylabel('Number of Ads', fontsize=11)
    axes[0, 0].set_title('Party Distribution (party_all)', fontsize=12, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3, axis='y')

# Party probability distributions
if 'prob_dem' in df.columns and 'prob_rep' in df.columns:
    axes[0, 1].hist(df['prob_dem'].dropna(), bins=50, alpha=0.5, label='Democrat', color='blue')
    axes[0, 1].hist(df['prob_rep'].dropna(), bins=50, alpha=0.5, label='Republican', color='red')
    axes[0, 1].set_xlabel('Probability', fontsize=11)
    axes[0, 1].set_ylabel('Frequency', fontsize=11)
    axes[0, 1].set_title('Party Probability Distributions', fontsize=12, fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

# Probability scatter plot
if 'prob_dem' in df.columns and 'prob_rep' in df.columns:
    sample_df = df[df['prob_dem'].notna() & df['prob_rep'].notna()].sample(min(5000, len(df)))
    axes[1, 0].scatter(sample_df['prob_dem'], sample_df['prob_rep'], alpha=0.3, s=1)
    axes[1, 0].set_xlabel('P(Democrat)', fontsize=11)
    axes[1, 0].set_ylabel('P(Republican)', fontsize=11)
    axes[1, 0].set_title('Democrat vs Republican Probabilities', fontsize=12, fontweight='bold')
    axes[1, 0].plot([0, 1], [1, 0], 'k--', alpha=0.5)
    axes[1, 0].grid(True, alpha=0.3)

# Party by spending
if 'party_all' in df.columns and 'spend_mid' in df.columns:
    party_spend = df.groupby('party_all')['spend_mid'].agg(['mean', 'median', 'sum'])
    axes[1, 1].bar(party_spend.index, party_spend['mean'], color=['blue', 'red', 'gray'], alpha=0.7)
    axes[1, 1].set_xlabel('Party', fontsize=11)
    axes[1, 1].set_ylabel('Mean Spending ($)', fontsize=11)
    axes[1, 1].set_title('Mean Spending by Party', fontsize=12, fontweight='bold')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
# Goal predictions analysis
print("=" * 80)
print("GOAL PREDICTIONS ANALYSIS")
print("=" * 80)

goal_cols = ['goal_DONATE_prediction', 'goal_PRIMARY_PERSUADE_prediction']
goal_prob_cols = ['goal_DONATE_predicted_prob', 'goal_PRIMARY_PERSUADE_predicted_prob']

for col in goal_cols:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts())
        print(f"  Missing: {df[col].isna().sum():,}")

for col in goal_prob_cols:
    if col in df.columns:
        print(f"\n{col} Statistics:")
        print(f"  Mean: {df[col].mean():.4f}")
        print(f"  Median: {df[col].median():.4f}")
        print(f"  Missing: {df[col].isna().sum():,}")


In [ ]:
# Visualize goal predictions
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Donate goal predictions
if 'goal_DONATE_prediction' in df.columns:
    donate_counts = df['goal_DONATE_prediction'].value_counts()
    axes[0, 0].bar(donate_counts.index.astype(str), donate_counts.values, color='green', alpha=0.7)
    axes[0, 0].set_xlabel('Donate Prediction', fontsize=11)
    axes[0, 0].set_ylabel('Number of Ads', fontsize=11)
    axes[0, 0].set_title('Donate Goal Predictions', fontsize=12, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3, axis='y')

# Primary persuade predictions
if 'goal_PRIMARY_PERSUADE_prediction' in df.columns:
    persuade_counts = df['goal_PRIMARY_PERSUADE_prediction'].value_counts()
    axes[0, 1].bar(persuade_counts.index.astype(str), persuade_counts.values, color='orange', alpha=0.7)
    axes[0, 1].set_xlabel('Persuade Prediction', fontsize=11)
    axes[0, 1].set_ylabel('Number of Ads', fontsize=11)
    axes[0, 1].set_title('Primary Persuade Goal Predictions', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3, axis='y')

# Donate probability distribution
if 'goal_DONATE_predicted_prob' in df.columns:
    axes[1, 0].hist(df['goal_DONATE_predicted_prob'].dropna(), bins=50, color='green', alpha=0.7, edgecolor='black')
    axes[1, 0].set_xlabel('Probability', fontsize=11)
    axes[1, 0].set_ylabel('Frequency', fontsize=11)
    axes[1, 0].set_title('Donate Probability Distribution', fontsize=12, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)

# Persuade probability distribution
if 'goal_PRIMARY_PERSUADE_predicted_prob' in df.columns:
    axes[1, 1].hist(df['goal_PRIMARY_PERSUADE_predicted_prob'].dropna(), bins=50, color='orange', alpha=0.7, edgecolor='black')
    axes[1, 1].set_xlabel('Probability', fontsize=11)
    axes[1, 1].set_ylabel('Frequency', fontsize=11)
    axes[1, 1].set_title('Persuade Probability Distribution', fontsize=12, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Tone and sentiment analysis
print("=" * 80)
print("TONE AND SENTIMENT ANALYSIS")
print("=" * 80)

tone_cols = ['ad_tone_constructed', 'ad_tone_mentionbased', 'tone_detected_entities']
sentiment_cols = ['ABSA_predicted_sentiment_agg']

for col in tone_cols:
    if col in df.columns:
        print(f"\n{col}:")
        value_counts = df[col].value_counts()
        print(value_counts.head(10))
        print(f"  Unique values: {df[col].nunique()}")
        print(f"  Missing: {df[col].isna().sum():,} ({df[col].isna().sum()/len(df)*100:.1f}%)")

for col in sentiment_cols:
    if col in df.columns:
        print(f"\n{col}:")
        value_counts = df[col].value_counts()
        print(value_counts)
        print(f"  Missing: {df[col].isna().sum():,}")


In [ ]:
# Visualize tone and sentiment
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Ad tone constructed
if 'ad_tone_constructed' in df.columns:
    tone_counts = df['ad_tone_constructed'].value_counts().head(10)
    axes[0, 0].barh(range(len(tone_counts)), tone_counts.values, color='steelblue', alpha=0.7)
    axes[0, 0].set_yticks(range(len(tone_counts)))
    axes[0, 0].set_yticklabels(tone_counts.index)
    axes[0, 0].set_xlabel('Count', fontsize=11)
    axes[0, 0].set_title('Top 10 Ad Tones (Constructed)', fontsize=12, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3, axis='x')

# Ad tone mention-based
if 'ad_tone_mentionbased' in df.columns:
    tone_mention_counts = df['ad_tone_mentionbased'].value_counts().head(10)
    axes[0, 1].barh(range(len(tone_mention_counts)), tone_mention_counts.values, color='coral', alpha=0.7)
    axes[0, 1].set_yticks(range(len(tone_mention_counts)))
    axes[0, 1].set_yticklabels(tone_mention_counts.index)
    axes[0, 1].set_xlabel('Count', fontsize=11)
    axes[0, 1].set_title('Top 10 Ad Tones (Mention-based)', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3, axis='x')

# Sentiment distribution
if 'ABSA_predicted_sentiment_agg' in df.columns:
    sentiment_counts = df['ABSA_predicted_sentiment_agg'].value_counts()
    axes[1, 0].bar(sentiment_counts.index.astype(str), sentiment_counts.values, color='green', alpha=0.7)
    axes[1, 0].set_xlabel('Sentiment', fontsize=11)
    axes[1, 0].set_ylabel('Count', fontsize=11)
    axes[1, 0].set_title('Sentiment Distribution', fontsize=12, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3, axis='y')

# Tone by party
if 'ad_tone_constructed' in df.columns and 'party_all' in df.columns:
    tone_party = df.groupby(['party_all', 'ad_tone_constructed']).size().unstack(fill_value=0)
    top_tones = df['ad_tone_constructed'].value_counts().head(5).index
    tone_party_top = tone_party[top_tones]
    tone_party_top.plot(kind='bar', ax=axes[1, 1], alpha=0.7)
    axes[1, 1].set_xlabel('Party', fontsize=11)
    axes[1, 1].set_ylabel('Count', fontsize=11)
    axes[1, 1].set_title('Top 5 Tones by Party', fontsize=12, fontweight='bold')
    axes[1, 1].legend(title='Tone', bbox_to_anchor=(1.05, 1), loc='upper left')
    axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
# Issue classification analysis
print("=" * 80)
print("ISSUE CLASSIFICATION ANALYSIS")
print("=" * 80)

issue_cols = ['issue_field', 'issue_class']

for col in issue_cols:
    if col in df.columns:
        print(f"\n{col}:")
        value_counts = df[col].value_counts()
        print(value_counts.head(20))
        print(f"  Unique values: {df[col].nunique()}")
        print(f"  Missing: {df[col].isna().sum():,} ({df[col].isna().sum()/len(df)*100:.1f}%)")


In [ ]:
# Visualize issue classifications
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# Issue field distribution
if 'issue_field' in df.columns:
    issue_counts = df['issue_field'].value_counts().head(15)
    axes[0].barh(range(len(issue_counts)), issue_counts.values, color='purple', alpha=0.7)
    axes[0].set_yticks(range(len(issue_counts)))
    axes[0].set_yticklabels(issue_counts.index)
    axes[0].set_xlabel('Count', fontsize=11)
    axes[0].set_title('Top 15 Issue Fields', fontsize=12, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='x')

# Issue class distribution
if 'issue_class' in df.columns:
    issue_class_counts = df['issue_class'].value_counts().head(15)
    axes[1].barh(range(len(issue_class_counts)), issue_class_counts.values, color='teal', alpha=0.7)
    axes[1].set_yticks(range(len(issue_class_counts)))
    axes[1].set_yticklabels(issue_class_counts.index)
    axes[1].set_xlabel('Count', fontsize=11)
    axes[1].set_title('Top 15 Issue Classes', fontsize=12, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()


In [ ]:
# Geographic and demographic targeting analysis
print("=" * 80)
print("GEOGRAPHIC AND DEMOGRAPHIC TARGETING ANALYSIS")
print("=" * 80)

# Helper function to parse region distribution
def parse_region_distribution(value):
    """Extract top regions from JSON format"""
    if pd.isna(value):
        return []
    try:
        if isinstance(value, str):
            data = json.loads(value)
            regions = [(item.get('region', ''), float(item.get('percentage', 0))) for item in data]
            return sorted(regions, key=lambda x: x[1], reverse=True)[:5]  # Top 5
    except:
        pass
    return []

# Helper function to parse demographic distribution
def parse_demographic_distribution(value):
    """Extract demographic info from JSON format"""
    if pd.isna(value):
        return {}
    try:
        if isinstance(value, str):
            data = json.loads(value)
            demo_dict = {}
            for item in data:
                age = item.get('age', '')
                gender = item.get('gender', '')
                percentage = float(item.get('percentage', 0))
                key = f"{age}_{gender}"
                demo_dict[key] = percentage
            return demo_dict
    except:
        pass
    return {}

# Analyze top regions
print("\nAnalyzing region distributions...")
all_regions = []
for idx, row in df.head(10000).iterrows():  # Sample for performance
    regions = parse_region_distribution(row.get('region_distribution', ''))
    all_regions.extend([r[0] for r in regions])

region_counts = Counter(all_regions)
print(f"\nTop 20 targeted regions:")
for region, count in region_counts.most_common(20):
    print(f"  {region}: {count:,}")

# Analyze demographics
print("\nAnalyzing demographic distributions...")
demo_keys = set()
for idx, row in df.head(10000).iterrows():  # Sample for performance
    demo = parse_demographic_distribution(row.get('demographic_distribution', ''))
    demo_keys.update(demo.keys())

print(f"\nUnique demographic segments: {len(demo_keys)}")
print(f"Sample segments: {list(demo_keys)[:10]}")


In [ ]:
# Publisher platforms analysis
print("=" * 80)
print("PUBLISHER PLATFORMS ANALYSIS")
print("=" * 80)

if 'publisher_platforms' in df.columns:
    # Parse publisher platforms (JSON array)
    def parse_platforms(value):
        if pd.isna(value):
            return []
        try:
            if isinstance(value, str):
                return json.loads(value)
        except:
            pass
        return []
    
    df['platforms_list'] = df['publisher_platforms'].apply(parse_platforms)
    all_platforms = []
    for platforms in df['platforms_list']:
        all_platforms.extend(platforms)
    
    platform_counts = Counter(all_platforms)
    print("\nPlatform distribution:")
    for platform, count in platform_counts.most_common():
        print(f"  {platform}: {count:,} ({count/len(df)*100:.1f}%)")
    
    # Media type
    if 'wmp_media_type' in df.columns:
        print(f"\nMedia Type distribution:")
        print(df['wmp_media_type'].value_counts())
        print(f"  Missing: {df['wmp_media_type'].isna().sum():,}")


In [ ]:
# Visualize platforms and media types
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Platform distribution
if 'platforms_list' in df.columns:
    platform_counts = Counter(all_platforms)
    platforms_df = pd.DataFrame(list(platform_counts.items()), columns=['Platform', 'Count'])
    platforms_df = platforms_df.sort_values('Count', ascending=True)
    
    axes[0].barh(platforms_df['Platform'], platforms_df['Count'], color='steelblue', alpha=0.7)
    axes[0].set_xlabel('Number of Ads', fontsize=11)
    axes[0].set_title('Publisher Platform Distribution', fontsize=12, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='x')

# Media type distribution
if 'wmp_media_type' in df.columns:
    media_counts = df['wmp_media_type'].value_counts()
    axes[1].bar(media_counts.index.astype(str), media_counts.values, color='coral', alpha=0.7)
    axes[1].set_xlabel('Media Type', fontsize=11)
    axes[1].set_ylabel('Count', fontsize=11)
    axes[1].set_title('Media Type Distribution', fontsize=12, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


In [ ]:
# Race of focus analysis
print("=" * 80)
print("RACE OF FOCUS ANALYSIS")
print("=" * 80)

if 'race_of_focus' in df.columns:
    print("\nRace of focus distribution:")
    race_counts = df['race_of_focus'].value_counts()
    print(race_counts)
    print(f"\n  Missing: {df['race_of_focus'].isna().sum():,} ({df['race_of_focus'].isna().sum()/len(df)*100:.1f}%)")
    
    if 'race_of_focus_region_pct' in df.columns:
        print(f"\nRace of focus region percentage statistics:")
        print(f"  Mean: {df['race_of_focus_region_pct'].mean():.4f}")
        print(f"  Median: {df['race_of_focus_region_pct'].median():.4f}")
        print(f"  Missing: {df['race_of_focus_region_pct'].isna().sum():,}")

# Visualize race of focus
if 'race_of_focus' in df.columns:
    plt.figure(figsize=(10, 6))
    race_counts = df['race_of_focus'].value_counts()
    plt.bar(race_counts.index.astype(str), race_counts.values, color='steelblue', alpha=0.7)
    plt.xlabel('Race of Focus', fontsize=11)
    plt.ylabel('Count', fontsize=11)
    plt.title('Race of Focus Distribution', fontsize=12, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()


In [ ]:
# Entity detection analysis
print("=" * 80)
print("ENTITY DETECTION ANALYSIS")
print("=" * 80)

entity_cols = ['detected_entities', 'detected_entities_federal', 'ABSA_detected_entities', 
                'combined_entities_federal']

for col in entity_cols:
    if col in df.columns:
        print(f"\n{col}:")
        # Count non-null values
        non_null = df[col].notna().sum()
        print(f"  Non-null: {non_null:,} ({non_null/len(df)*100:.1f}%)")
        
        # Sample some values
        sample_values = df[col].dropna().head(5)
        if len(sample_values) > 0:
            print(f"  Sample values:")
            for val in sample_values:
                print(f"    - {str(val)[:100]}")

# ABSA number of mentions
if 'ABSA_number_of_mentions' in df.columns:
    print(f"\nABSA_number_of_mentions Statistics:")
    print(f"  Mean: {df['ABSA_number_of_mentions'].mean():.2f}")
    print(f"  Median: {df['ABSA_number_of_mentions'].median():.2f}")
    print(f"  Max: {df['ABSA_number_of_mentions'].max():.0f}")
    print(f"  Missing: {df['ABSA_number_of_mentions'].isna().sum():,}")


In [ ]:
# Page and advertiser analysis
print("=" * 80)
print("PAGE AND ADVERTISER ANALYSIS")
print("=" * 80)

if 'page_name' in df.columns:
    print(f"\nUnique pages: {df['page_name'].nunique():,}")
    print(f"\nTop 20 pages by ad count:")
    page_counts = df['page_name'].value_counts().head(20)
    print(page_counts)

if 'disclaimer' in df.columns:
    print(f"\nUnique disclaimers: {df['disclaimer'].nunique():,}")
    print(f"\nTop 20 disclaimers by ad count:")
    disclaimer_counts = df['disclaimer'].value_counts().head(20)
    print(disclaimer_counts)

# Visualize top pages
if 'page_name' in df.columns:
    plt.figure(figsize=(12, 8))
    top_pages = df['page_name'].value_counts().head(15)
    plt.barh(range(len(top_pages)), top_pages.values, color='steelblue', alpha=0.7)
    plt.yticks(range(len(top_pages)), top_pages.index)
    plt.xlabel('Number of Ads', fontsize=11)
    plt.title('Top 15 Pages by Ad Count', fontsize=12, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()


In [ ]:
# Correlation analysis for key numerical features
print("=" * 80)
print("CORRELATION ANALYSIS")
print("=" * 80)

# Select numerical columns for correlation
numerical_cols = [
    'spend_mid', 'impressions_mid', 'cpm', 'ad_duration_days',
    'prob_dem', 'prob_rep', 'prob_other',
    'goal_DONATE_predicted_prob', 'goal_PRIMARY_PERSUADE_predicted_prob',
    'ABSA_number_of_mentions', 'race_of_focus_region_pct'
]

# Filter to columns that exist
numerical_cols = [col for col in numerical_cols if col in df.columns]

corr_matrix = df[numerical_cols].corr()

# Visualize correlation matrix
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Key Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print strong correlations
print("\nStrong correlations (|r| > 0.5):")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > 0.5:
            print(f"  {corr_matrix.columns[i]} <-> {corr_matrix.columns[j]}: {corr_val:.3f}")


In [ ]:
# Feature engineering for manipulation detection
print("=" * 80)
print("FEATURE ENGINEERING FOR MANIPULATION DETECTION")
print("=" * 80)

# Create potential manipulation indicators
df['high_spend'] = (df['spend_mid'] > df['spend_mid'].quantile(0.95)).astype(int)
df['high_impressions'] = (df['impressions_mid'] > df['impressions_mid'].quantile(0.95)).astype(int)
df['long_duration'] = (df['ad_duration_days'] > df['ad_duration_days'].quantile(0.95)).astype(int)
df['high_persuade_prob'] = (df['goal_PRIMARY_PERSUADE_predicted_prob'] > 0.7).astype(int) if 'goal_PRIMARY_PERSUADE_predicted_prob' in df.columns else 0
df['extreme_party_prob'] = ((df['prob_dem'] > 0.9) | (df['prob_rep'] > 0.9)).astype(int) if 'prob_dem' in df.columns else 0

# Check for missing critical information (potential red flag)
df['missing_spend'] = df['spend_mid'].isna().astype(int)
df['missing_impressions'] = df['impressions_mid'].isna().astype(int)
df['missing_party'] = df['party_all'].isna().astype(int) if 'party_all' in df.columns else 0

print("\nManipulation indicator features created:")
manipulation_features = ['high_spend', 'high_impressions', 'long_duration', 
                         'high_persuade_prob', 'extreme_party_prob',
                         'missing_spend', 'missing_impressions', 'missing_party']

for feat in manipulation_features:
    if feat in df.columns:
        print(f"  {feat}: {df[feat].sum():,} ({df[feat].mean()*100:.1f}%)")


In [ ]:
# Analyze patterns that might indicate manipulation
print("=" * 80)
print("PATTERNS POTENTIALLY INDICATIVE OF MANIPULATION")
print("=" * 80)

# 1. Ads with high persuade probability but low transparency
if 'goal_PRIMARY_PERSUADE_predicted_prob' in df.columns:
    high_persuade = df[df['goal_PRIMARY_PERSUADE_predicted_prob'] > 0.8]
    print(f"\n1. High persuade probability ads (>0.8): {len(high_persuade):,}")
    if len(high_persuade) > 0:
        print(f"   - Mean spending: ${high_persuade['spend_mid'].mean():,.2f}")
        print(f"   - Mean impressions: {high_persuade['impressions_mid'].mean():,.0f}")
        if 'party_all' in df.columns:
            print(f"   - Party distribution:")
            print(high_persuade['party_all'].value_counts())

# 2. Ads with extreme party probabilities
if 'prob_dem' in df.columns and 'prob_rep' in df.columns:
    extreme_dem = df[df['prob_dem'] > 0.95]
    extreme_rep = df[df['prob_rep'] > 0.95]
    print(f"\n2. Extreme party probability ads:")
    print(f"   - Extreme Democrat (>0.95): {len(extreme_dem):,}")
    print(f"   - Extreme Republican (>0.95): {len(extreme_rep):,}")

# 3. Ads with missing critical information
print(f"\n3. Ads with missing critical information:")
print(f"   - Missing spending: {df['missing_spend'].sum():,} ({df['missing_spend'].mean()*100:.1f}%)")
print(f"   - Missing impressions: {df['missing_impressions'].sum():,} ({df['missing_impressions'].mean()*100:.1f}%)")
if 'party_all' in df.columns:
    print(f"   - Missing party: {df['missing_party'].sum():,} ({df['missing_party'].mean()*100:.1f}%)")

# 4. High spending, high impressions, high persuade
if 'goal_PRIMARY_PERSUADE_predicted_prob' in df.columns:
    high_all = df[
        (df['spend_mid'] > df['spend_mid'].quantile(0.9)) &
        (df['impressions_mid'] > df['impressions_mid'].quantile(0.9)) &
        (df['goal_PRIMARY_PERSUADE_predicted_prob'] > 0.7)
    ]
    print(f"\n4. High spend + High impressions + High persuade: {len(high_all):,}")
    if len(high_all) > 0:
        print(f"   - Mean spending: ${high_all['spend_mid'].mean():,.2f}")
        print(f"   - Mean impressions: {high_all['impressions_mid'].mean():,.0f}")


In [ ]:
# Summary statistics for key features
print("=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

summary_cols = ['spend_mid', 'impressions_mid', 'cpm', 'ad_duration_days',
                'prob_dem', 'prob_rep', 'prob_other',
                'goal_DONATE_predicted_prob', 'goal_PRIMARY_PERSUADE_predicted_prob']

summary_cols = [col for col in summary_cols if col in df.columns]
print(df[summary_cols].describe())


## Key Findings and Insights

### Data Quality
- Dataset contains **377,722** ads with **60+ features**
- Significant missing data in some columns (entity detection, sentiment analysis)
- JSON-formatted fields (spending, impressions, geographic/demographic targeting) need parsing

### Spending and Reach
- Wide range of spending and impressions (highly skewed distributions)
- Most ads have relatively low spending, with a few high-spending outliers
- Cost per mille (CPM) varies significantly

### Temporal Patterns
- Ads created throughout 2022
- Potential seasonal patterns in ad creation
- Ad duration varies widely

### Political Characteristics
- Party classifications (Democrat, Republican, Other)
- Probability scores for party affiliation
- Goal predictions (Donate, Primary Persuade)

### Content Analysis
- Tone detection (constructed and mention-based)
- Sentiment analysis (ABSA)
- Issue classifications (field and class)
- Entity detection

### Targeting
- Geographic targeting (region distribution)
- Demographic targeting (age, gender)
- Platform distribution (Facebook, Instagram)

### Features for Manipulation Detection
1. **High persuade probability** combined with high spending
2. **Extreme party probabilities** (very high Democrat or Republican scores)
3. **Missing transparency information** (spending, impressions, party)
4. **Aggressive targeting** (narrow demographic/geographic focus)
5. **Tone and sentiment patterns** (negative, attack-oriented content)
6. **Issue classification patterns** (focus on divisive issues)

## Recommendations for Model Development

1. **Feature Engineering:**
   - Parse JSON fields (spending, impressions, targeting)
   - Create interaction features (spend × impressions × persuade_prob)
   - Extract temporal features (day of week, month, proximity to elections)
   - Create targeting concentration metrics

2. **Target Variable:**
   - Define manipulation based on domain expertise
   - Consider multiple manipulation types (misinformation, emotional manipulation, targeting manipulation)

3. **Data Preprocessing:**
   - Handle missing values appropriately
   - Normalize/standardize numerical features
   - Encode categorical features
   - Handle class imbalance if present

4. **Model Considerations:**
   - Use ensemble methods (Random Forest, XGBoost)
   - Consider deep learning for text features
   - Implement feature importance analysis
   - Use cross-validation for robust evaluation
